# Emissions-weighted carbon price

# Packages and libraries

In [1]:
import pandas as pd
import numpy as np
import os
os.environ["WCPD_SKIP_PULL"] = "1"
import re
import requests
import subprocess
from pathlib import Path
import importlib.util
import sys, dep_ecp
import os
from pathlib import Path
from itertools import chain
from zipfile import ZipFile
from io import BytesIO

path_aux_data = Path('/Users/geoffroydolphin/OneDrive - rff/documents/research/projects/ecp/ecp_dataset')

from dep_ecp import ecp_v3_gen_func as ecp_general
from dep_ecp import ecp_v3_coverageFactors as ecp_cov_fac
from dep_ecp import inventory_preproc_nat as ecp_inv_nat
from dep_ecp import inventory_preproc_subnat as ecp_inv_subnat
from dep_ecp import ecp_v3_inventory_share_func as ecp_inv_share
from dep_ecp import ecp_v3_coverage as ecp_coverage
from dep_ecp import ecp_v3_currConversion as ecp_cur_conv
from dep_ecp import ecp_v3_overlap as ecp_overlap
from dep_ecp import ecp_v3_weightedAverage as ecp_wav
from dep_ecp import ecp_v3_aggSec as ecp_sec_em

importlib.reload(ecp_general)
importlib.reload(ecp_cur_conv)
importlib.reload(ecp_inv_share)
importlib.reload(ecp_coverage)  # child first
importlib.reload(dep_ecp)       # then parent (if needed)

from wcpd_utils import jurisdictions as jur_loader

print(sys.executable)           # should be 3.13 python
print(dep_ecp.__file__)  

from datetime import date
today = date.today()
d1 = today.strftime("%b-%d-%Y")

def find_project_root(markers=("pyproject.toml","setup.cfg","requirements.txt",".git",".project-root")):
    p = Path.cwd().resolve()
    for parent in (p, *p.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    return p

REPO_ROOT = find_project_root()

def resolve_wcpd_repo_root():
    candidates = []
    env_root = os.environ.get("WCPD_REPO_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([
        REPO_ROOT.parent / "WorldCarbonPricingDatabase",
        Path.home() / "GitHub" / "WorldCarbonPricingDatabase",
    ])
    for candidate in candidates:
        if (candidate / "_dataset" / "data").exists() and (candidate / "_raw" / "overlap").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the WorldCarbonPricingDatabase checkout. "
        "Set WCPD_REPO_ROOT to the repo root."
    )

def sync_wcpd_repo(repo_root):
    if os.environ.get("WCPD_SKIP_PULL") == "1":
        print("Skipping WCPD sync because WCPD_SKIP_PULL=1.")
        return
    if not (repo_root / ".git").exists():
        print(f"Skipping WCPD sync because {repo_root} is not a git checkout.")
        return
    status = subprocess.run(
        ["git", "-C", str(repo_root), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    )
    if status.stdout.strip():
        print("Skipping WCPD sync because the checkout has local changes.")
        return
    result = subprocess.run(
        ["git", "-C", str(repo_root), "pull", "--ff-only"],
        check=True,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())

WCPD_REPO_ROOT = resolve_wcpd_repo_root()
DB_VERSION = "v2026.1"
DB_VERSION_FALLBACKS = {"CH4": ["v2025.0"], "N2O": ["v2025.0"]}
sync_wcpd_repo(WCPD_REPO_ROOT)
path_wcpd = WCPD_REPO_ROOT / "_dataset" / "data" / DB_VERSION
path_ghg = REPO_ROOT / "_raw" / "ghg_inventory" / "raw"
path_ghg_processed = REPO_ROOT / "_raw" / "ghg_inventory" / "processed"
path_aux_files = REPO_ROOT / "_raw" / "_aux_files"
path_aux_data = REPO_ROOT / "_raw"
path_dataset_output = REPO_ROOT / "_output" / "_dataset" / DB_VERSION
edgar_subnat_output_root = path_ghg / "subnational" / "jrc_edgar_gridded" / "output"
ecp_inv_nat.path_ghg = str(path_ghg)

def resolve_wcpd_gas_path(gas):
    versions = [DB_VERSION] + DB_VERSION_FALLBACKS.get(gas, [])
    checked = []
    for version in versions:
        gas_path = WCPD_REPO_ROOT / "_dataset" / "data" / version / gas
        national = gas_path / "national"
        subnational = gas_path / "subnational"
        checked.append(str(gas_path))
        if national.exists() and subnational.exists() and any(national.glob("*.csv")) and any(subnational.glob("*.csv")):
            return gas_path, version
    raise FileNotFoundError(
        f"Could not find WCPD files for {gas}. Checked: {checked}"
    )

/Library/Frameworks/Python.framework/Versions/3.13/bin/python3
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/__init__.py
Skipping WCPD sync because WCPD_SKIP_PULL=1.


In [2]:
jurisdictions = jur_loader.jurisdictions

# Pick the top-level countries you want
countries = ["United States", "Canada", "China", "Mexico", "Japan"]

# Validate keys exist
missing = [c for c in countries if c not in jurisdictions.get("subnationals", {})]
if missing:
    raise KeyError(f"Missing subnational lists for: {missing}")

subnat_lists = {c: jurisdictions["subnationals"][c] for c in countries}
all_subnat_list = list(chain.from_iterable(subnat_lists[c] for c in countries))

gases = ["CO2", "CH4", "N2O"]  # extend as needed: , "FGASES"

def infer_subnat_last_inv_year(inventories_subnat, subnat_lists):
    years = {}
    for country, jurisdictions in subnat_lists.items():
        available = []
        for gas_df in inventories_subnat.values():
            subset = gas_df.loc[gas_df["jurisdiction"].isin(jurisdictions), "year"]
            if not subset.empty:
                available.append(int(subset.max()))
        if not available:
            raise ValueError(f"No subnational inventory years found for {country}.")
        years[country] = min(available)
    return years

def extend_totals_to_year(df, end_year, group_cols):
    if df.empty:
        return df
    max_year = int(df["year"].max())
    if max_year >= end_year:
        return df
    last_rows = df.loc[df["year"] == max_year].copy()
    frames = [df]
    for year in range(max_year + 1, end_year + 1):
        temp = last_rows.copy()
        temp["year"] = year
        frames.append(temp)
    return pd.concat(frames, ignore_index=True)

lastInvYear = {"national": 2022, "subnat": {}}
lastDbYear = 2025


# Institutional design (World Carbon Pricing Database)

In [3]:
wcpd = {}

# Load IPCC to IEA mapping once
ipcc_iea_map = pd.read_csv(
    REPO_ROOT / Path("_raw/_aux_files/ipcc2006_iea_category_codes.csv"),
    usecols=["ipcc_code", "FLOW"]
).rename(columns={"FLOW": "iea_code"})

for gas in gases:
    gas_path, gas_version = resolve_wcpd_gas_path(gas)
    print(f"Building policy features data frame for {gas} from {gas_version}")

    # Load WCPD data
    wcpd_ctry = ecp_general.concatenate(str(gas_path / "national"))
    wcpd_subnat = ecp_general.concatenate(str(gas_path / "subnational"))
    wcpd_all = pd.concat([wcpd_ctry, wcpd_subnat]).sort_values(by=["jurisdiction", "year"])

    # Clean and deduplicate
    wcpd_all["Product"] = wcpd_all["Product"].fillna('NA')
    wcpd_all = wcpd_all.drop_duplicates(subset=["jurisdiction", "year", "ipcc_code", "Product"])

    # Add IEA sector codes
    wcpd_all = wcpd_all.merge(ipcc_iea_map, on="ipcc_code", how="left")
    wcpd_all["iea_code"] = wcpd_all["iea_code"].fillna('NA')

    # Safeguard to ensure column types
    cols = ["ets_2_id", "tax_2_id", "ets_2_curr_code"]
    wcpd_all = wcpd_all.astype({c: "string" for c in cols if c in wcpd_all.columns})

    # Standardize jurisdiction names
    def standardize(names):
        return {name: name.replace(".", "").replace(",", "").replace(" ", "_") for name in names}

    ctry_names = wcpd_ctry["jurisdiction"].unique()
    subnat_names = wcpd_subnat["jurisdiction"].unique()
    countries_dic = standardize(ctry_names)
    subnat_dic = standardize(subnat_names)

    # Check for duplicates
    if wcpd_all.duplicated(["jurisdiction", "year", "ipcc_code", "Product"]).any():
        print(f"The dataset for {gas} contains duplicates!")

    # Add coverage factors
    wcpd_all = ecp_cov_fac.coverageFactors(wcpd_all, gas, WCPD_REPO_ROOT)

    # Handle mechanism overlaps
    overlap = pd.read_csv(WCPD_REPO_ROOT / "_raw" / "overlap" / f"overlap_mechanisms_{gas}.csv")
    wcpd_all = ecp_overlap.overlap(wcpd_all, overlap)

    # Save
    wcpd[gas] = wcpd_all


Building policy features data frame for CO2 from v2026.1


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_coverageFactors.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  coverageFactor = pd.concat([pd.read_csv(path) for path in coverage_files], ignore_index=True)


coverageFactor has 355950 duplicate keys for CO2; keeping the last row per key from sorted files (3058 keys had conflicting cf values).


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]] = 0
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]+"_ids"] = inst_df_ids.loc[:, scheme_columns[i[0]]] + inst_df_ids.loc[:, scheme_columns[i[1]]]
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_ut

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]+"_ids"] = inst_df_ids.loc[:, scheme_columns[i[0]]] + inst_df_ids.loc[:, scheme_columns[i[1]]]
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]] = 0


...

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]] = 0
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]+"_ids"] = inst_df_ids.loc[:, scheme_columns[i[0]]] + inst_df_ids.loc[:, scheme_columns[i[1]]]
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_ut

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]+"_ids"] = inst_df_ids.loc[:, scheme_columns[i[0]]] + inst_df_ids.loc[:, scheme_columns[i[1]]]
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]] = 0


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.loc[:, "overlap_"+i[0]+"_"+i[1]+"_ids"] = inst_df_ids.loc[:, scheme_columns[i[0]]] + inst_df_ids.loc[:, scheme_columns[i[1]]]


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.drop(ovp_columns[ovp_col], axis=1, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.drop(ovp_columns[ovp_col], axis=1, inplace=True)
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_overlap.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  inst_df_ids.drop(ovp_columns[ovp_col], axis=1, inplace=True)


# Emissions
## I. National jurisdictions 
### I.A Total GHG emissions (EDGAR)

In [4]:
# Global Warming Potential values
ipcc_gwp = pd.read_csv(REPO_ROOT / "_raw/ghg_inventory/gwp_list.csv")
ipcc_gwp_list = dict(zip(ipcc_gwp.edgar_label, ipcc_gwp.ar5_gwp_100y))

In [5]:
EDGAR_URL = "https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/EDGAR/datasets/EDGAR_2025_GHG/"

edgar_files = {
    "CH4": {
        "zip": ["EDGAR_CH4_1970_2024.zip", "EDGAR_CH4_1970_2023.zip"],
        "xlsx": ["EDGAR_CH4_1970_2024.xlsx", "EDGAR_CH4_1970_2023.xlsx"],
    },
    "CO2": {
        "zip": ["IEA_EDGAR_CO2_1970_2024.zip", "IEA_EDGAR_CO2_1970_2023.zip"],
        "xlsx": ["IEA_EDGAR_CO2_1970_2024.xlsx", "IEA_EDGAR_CO2_1970_2023.xlsx"],
    },
    "FGASES": {
        "zip": ["EDGAR_F-gases_1990_2024.zip", "EDGAR_F-gases_1990_2023.zip"],
        "xlsx": ["EDGAR_F-gases_1990_2024.xlsx", "EDGAR_F-gases_1990_2023.xlsx"],
    },
    "N2O": {
        "zip": ["EDGAR_N2O_1970_2024.zip", "EDGAR_N2O_1970_2023.zip"],
        "xlsx": ["EDGAR_N2O_1970_2024.xlsx", "EDGAR_N2O_1970_2023.xlsx"],
    },
}

sheetNames = {"CH4":"IPCC 2006",
              "CO2":"IPCC 2006",
              "FGASES":"IPCC 2006",
              "N2O":"IPCC 2006"}

def resolve_zip_member(zip_file, candidates):
    members = zip_file.namelist()
    for candidate in candidates:
        match = next((name for name in members if name.endswith(candidate)), None)
        if match is not None:
            return match
    raise FileNotFoundError(f"Could not find any of {candidates} in archive members: {members[:10]}")

edgar_ghg = {}
cached_ipcc_totals = path_ghg_processed / "ghg_national_total_ipcc.csv"

if cached_ipcc_totals.exists():
    print(f"Using cached processed EDGAR inventories from {cached_ipcc_totals}")
else:
    for gas, file_cfg in edgar_files.items():
        print(gas)
        response = None
        resolved_url = None
        for zip_name in file_cfg["zip"]:
            candidate_url = EDGAR_URL + zip_name
            response = requests.get(candidate_url, verify=False)
            if response.ok:
                resolved_url = candidate_url
                break
        if response is None or not response.ok:
            raise FileNotFoundError(f"No EDGAR archive found for {gas} under {EDGAR_URL}: {file_cfg['zip']}")
        print(f"Using {resolved_url}")

        ## Open zip folder
        myzip = ZipFile(BytesIO(response.content))
        
        download = myzip.open(resolve_zip_member(myzip, file_cfg["xlsx"]))

        df = pd.read_excel(download, header=0,
                        sheet_name=sheetNames[gas], skiprows=[x for x in range(0,9)])
        
        edgar_ghg[gas] = df

Using cached processed EDGAR inventories from /Users/geoffroydolphin/GitHub/ECP/_raw/ghg_inventory/processed/ghg_national_total_ipcc.csv


In [6]:
cached_ipcc_totals = path_ghg_processed / "ghg_national_total_ipcc.csv"
cached_nat_totals = path_ghg_processed / "ghg_national_total.csv"

if cached_ipcc_totals.exists() and cached_nat_totals.exists():
    edgar_wb_map = {}
    df_gases = pd.read_csv(cached_ipcc_totals)
    df_gases_jurAgg = pd.read_csv(cached_nat_totals)
    if df_gases_jurAgg["year"].max() < 2023:
        df_gases_jurAgg = df_gases.groupby(["jurisdiction", "year"]).sum().reset_index()
        df_gases_jurAgg.drop(["ipcc_code", "Substance"], axis=1, inplace=True, errors="ignore")
        path_ghg_processed.mkdir(parents=True, exist_ok=True)
        df_gases_jurAgg.to_csv(path_ghg_processed / "ghg_national_total.csv", index=None)
else:
    # concordance between EDGAR and World Bank country names
    edgar_wb_map_path = path_aux_files / "edgar_wb_ctry_name_map.csv"
    if edgar_wb_map_path.exists():
        edgar_wb_map = pd.read_csv(edgar_wb_map_path)
        edgar_wb_map = edgar_wb_map.loc[~edgar_wb_map.ctry_name_wb.isnull()]
        edgar_wb_map = dict(zip(list(edgar_wb_map['ctry_name_edgar'].values), list(edgar_wb_map['ctry_name_wb'].values)))
    else:
        print(f"No EDGAR country-name concordance found at {edgar_wb_map_path}; using raw EDGAR names.")
        edgar_wb_map = {}

    def process_gas_dataframe(gas, df, gwp_dict, edgar_wb_map):
        df = df.copy()
        df = df[df["fossil_bio"] == "fossil"]
        df = df.drop(columns=["IPCC_annex", "C_group_IM24_sh", "Country_code_A3", "fossil_bio", "ipcc_code_2006_for_standard_report_name"])
        df.rename(columns={"Name": "jurisdiction"}, inplace=True)
        df = df.groupby(["jurisdiction", "ipcc_code_2006_for_standard_report", "Substance"]).sum().reset_index()
        df = df.melt(
            id_vars=["jurisdiction", "ipcc_code_2006_for_standard_report", "Substance"],
            var_name="year",
            value_name=gas
        )
        df["year"] = df["year"].str[2:].astype(int)
        df["jurisdiction"] = df["jurisdiction"].replace(edgar_wb_map)

        if gas in ["CO2", "CH4", "N2O"]:
            df[gas] = df[gas] * gwp_dict[gas]
            df.drop(columns=["Substance"], inplace=True)
        else:
            df_gwp = pd.DataFrame({"Substance": gwp_dict.keys(), "gwp": gwp_dict.values()})
            df = df.merge(df_gwp, on="Substance", how="left")
            df[gas] = df[gas] * df["gwp"]
            df.drop(columns=["gwp"], inplace=True)
            df = df.groupby(["jurisdiction", "year", "ipcc_code_2006_for_standard_report"]).sum().reset_index()

        df.rename(columns={"ipcc_code_2006_for_standard_report": "ipcc_code"}, inplace=True)
        df["ipcc_code"] = df["ipcc_code"].apply(lambda x: x.replace('.', '').upper())
        df["ipcc_code"] = df["ipcc_code"].apply(lambda x: x.replace('_NORES', '').upper())
        return df

    df_gases = pd.DataFrame()
    for gas, df in edgar_ghg.items():
        df_gas = process_gas_dataframe(gas, df, ipcc_gwp_list, edgar_wb_map)
        if df_gases.empty:
            df_gases = df_gas
        else:
            df_gases = df_gases.merge(df_gas, on=["jurisdiction", "year", "ipcc_code"], how="outer")

    df_gases["all_GHG"] = df_gases[list(edgar_ghg.keys())].sum(axis=1)
    df_gases_jurAgg = df_gases.groupby(["jurisdiction", "year"]).sum().reset_index()
    df_gases_jurAgg.drop(["ipcc_code", "Substance"], axis=1, inplace=True, errors="ignore")
    path_ghg_processed.mkdir(parents=True, exist_ok=True)
    df_gases_jurAgg.to_csv(path_ghg_processed / "ghg_national_total.csv", index=None)

df_gases_jurAgg = extend_totals_to_year(df_gases_jurAgg, lastDbYear, ["jurisdiction"])


In [7]:
# Country names
iea_wb_map = {'Australi':'Australia', 
            'Bosniaherz':'Bosnia and Herzegovina',
            'Brunei':'Brunei Darussalam', 
            'Congo':'Congo, Rep.', 
            'Congorep':'Congo, Dem. Rep.',
            'Costarica':'Costa Rica',
            'Coteivoire':"Cote d'Ivoire", 
            'Czech':'Czech Republic',
            'Dominicanr':'Dominican Republic',
            'Egypt':'Egypt, Arab Rep.', 
            'Elsalvador':'El Salvador',
            'Eqguinea':'Equatorial Guinea',
            'Eswatini':'Lesotho', 
            'Hongkong':'Hong Kong, SAR', 
            'Iran':'Iran, Islamic, Rep.', 
            'Korea':'Korea, Rep.', 
            'Koreadpr':'Korea, Dem. Rep.', 
            'Kyrgyzstan':'Kyrgyz Republic', 
            'Lao':'Lao PDR', 
            'Luxembou':'Luxembourg',
            'Nethland':'Netherlands',
            'Northmaced':'North Macedonia',
            'Nz':'New Zealand',
            'Philippine':'Philippines',
            'Russia':'Russian Federation',
            'Saudiarabi':'Saudi Arabia', 
            'Slovakia':'Slovak Republic',
            'Southafric':'South Africa',
            'Srilanka':'Sri Lanka', 
            'Ssudan':'South Sudan', 
            'Switland':'Switzerland', 
            'Syria':'Syrian Arab Republic', 
            'Turkmenist':'Turkmenistan',
            'Uae':'United Arab Emirates',
            'Uk':'United Kingdom',
            'Usa':'United States',
            'Venezuela':'Venezuela, RB',
            'Yemen':'Yemen, Rep.'}

In [8]:
# CREATE DATAFRAME WITH TOTAL EMISSIONS WORLD

df_gases_tot_world = df_gases_jurAgg.groupby(by=["year"]).sum()
df_gases_tot_world.reset_index(inplace=True)
df_gases_tot_world = extend_totals_to_year(df_gases_tot_world, lastDbYear, [])
path_ghg_processed.mkdir(parents=True, exist_ok=True)
df_gases_tot_world.to_csv(path_ghg_processed / "ghg_world_total.csv", index=None)

### I.B National GHG Inventory (kt and % of totals)

In [9]:
# Load inventories
inventories = {
    "CO2": ecp_inv_nat.inventory_co2(wcpd_all, ctry_names, iea_wb_map, df_gases, edgar_wb_map),
    "CH4": ecp_inv_nat.inventory_non_co2(wcpd_all, "CH4", ctry_names, iea_wb_map, edgar_wb_map),
    "N2O": ecp_inv_nat.inventory_non_co2(wcpd_all, "N2O", ctry_names, iea_wb_map, edgar_wb_map),
#    "FGASES": ecp_inv_nat.inventory_non_co2("FGASES", df_gases)
}

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["CO2"].replace({"..": np.nan, "x": np.nan, "c": np.nan}, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:62: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["jurisdiction"].replace(iea_wb_map, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:90: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  ippu_fug_nat["jurisdiction"].replace(edgar_wb_map, inplace=True)
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:90: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-cop

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:144: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["jurisdiction"].replace(to_replace=iea_wb_map, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:169: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  edgar_ghg["jurisdiction"].replace(edgar_wb_map, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:144: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["jurisdiction"].replace(to_replace=iea_wb_map, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_nat.py:169: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  edgar_ghg["jurisdiction"].replace(edgar_wb_map, inplace=True)


In [10]:
inventories_wldSect = {}

for gas in ["CO2", "CH4", "N2O"]:#, inventory in inventories.items()[0]:
    if gas == "CO2":
        inventory = inventories[gas][0]
        int_bunkers = inventories[gas][1]
    else:
        inventory = inventories[gas].copy()
        inventory.drop(['ipcc_code2', 'ipcc_code3', 'FLOWname', 'FLOW', 'Source'], axis=1, inplace=True)

    inventory = inventory[(inventory["year"]<=lastInvYear["national"])]

    # Ensure float data type for emissions column
    df = inventory
    df.loc[:, gas] = df[gas].replace('', np.nan)    
    df.loc[:, gas] = pd.to_numeric(df[gas], errors="coerce")
    inventories[gas] = df

    # Calculate shares
    inventory_share = ecp_inv_share.emissions_share(inventory, df_gases_jurAgg, df_gases_tot_world, gas)

    # Define merge keys and columns
    merge_keys = ["jurisdiction", "year", "ipcc_code", "iea_code", "Product"]
    value_columns = ["jurisdiction", "year", "ipcc_code", "iea_code", "Product", gas]

    # Merge shares into inventory
    inventory = inventory.merge(inventory_share, on=merge_keys, how="left")
    inventories[gas] = inventory

    # Save to CSV
    path_ghg_processed.mkdir(parents=True, exist_ok=True)
    out_path = path_ghg_processed / f"inventory_nat_{gas}.csv"
    inventory.to_csv(out_path, index=False)

    # Calculate world sector shares
    # need to add international marine and aviation bunkers to sector-specific totals (as they are not included in country-sector data)
    if gas == "CO2":
        sectors_wld_total = pd.concat([inventory[value_columns], int_bunkers])
        sectors_wld_total = sectors_wld_total.groupby(["ipcc_code", "year"], as_index=False).sum()
        sectors_wld_total.drop(["jurisdiction", "iea_code", "Product"], axis=1, inplace=True)

    else:
        sectors_wld_total = inventory[value_columns].copy()
        sectors_wld_total = sectors_wld_total.groupby(["ipcc_code", "year"], as_index=False).sum()
        sectors_wld_total.drop(["jurisdiction"], axis=1, inplace=True)

    inventories_wldSect[gas] = ecp_inv_share.emissions_share_wld_sectors(inventory, sectors_wld_total, gas, "national")


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_inventory_share_func.py:180: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[share_col] = merged[gas].astype(float).div(denom.replace({0: np.nan}))


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_inventory_share_func.py:180: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[share_col] = merged[gas].astype(float).div(denom.replace({0: np.nan}))


In [11]:
for gas in gases:
    gas_dir = path_ghg_processed / "national" / gas
    gas_dir.mkdir(parents=True, exist_ok=True)
    for ctry in countries_dic.keys():
        inventories[gas].loc[inventories[gas].jurisdiction==ctry, :].to_csv(gas_dir / f"inventory_{gas}_{countries_dic[ctry]}.csv", index=None)

## II. Subnational jurisdictions
### II.A Total GHG emissions

In [12]:
can = ecp_inv_subnat.load_canada_data(path_ghg)
chn = ecp_inv_subnat.load_china_data(path_ghg)
usa = ecp_inv_subnat.load_usa_data(path_ghg)
mex = ecp_inv_subnat.load_edgar_subnat_jpn_mex(edgar_subnat_output_root, countries=["Mexico"])
jpn = ecp_inv_subnat.load_edgar_subnat_jpn_mex(edgar_subnat_output_root, countries=["Japan"])

inventory_subnat = pd.concat([can, chn, usa, mex, jpn])
inventory_subnat = pd.merge(inventory_subnat, ipcc_iea_map, on=['ipcc_code'], how='left')

df_gases_tot_subnat = ecp_inv_subnat.generate_subnat_total(can, chn, usa, mex, jpn)
path_ghg_processed.mkdir(parents=True, exist_ok=True)
df_gases_tot_subnat.to_csv(path_ghg_processed / "ghg_subnat_total.csv", index=None)

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_subnat.py:63: DtypeWarning: Columns (5,6,8,9,10,11,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_subnat.py:75: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df.ipcc_code.replace(category_names_ipcc_can_map, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_subnat.py:167: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Gas"].replace({"CO2 (combustion)": "CO2", "CO2 (non-combustion)": "CO2"}, inplace=True)
/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/inventory_preproc_subnat.py:179: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will

### II.B Subnational inventory (kt and % totals)

In [13]:
inventories_subnat = {}
inventories_subnat_wldSect = {}
inventories_subnat_ctrySect = {}

for gas in gases:
    print(gas)

    inventory_subnat_gas = inventory_subnat[["supra_jur", "jurisdiction", "year", "ipcc_code", "iea_code", gas]]

    inventory_subnat_share = ecp_inv_share.emissions_share(inventory_subnat_gas, 
                                                           df_gases_tot_subnat, df_gases_tot_world, gas, df_gases_jurAgg, "subnational")

    merge_keys = ["supra_jur", "jurisdiction", "year", "ipcc_code", "iea_code"]
    columns = ["supra_jur", "jurisdiction", "year", "ipcc_code", gas]

    inventories_subnat[gas] = pd.merge(inventory_subnat_gas, inventory_subnat_share, on=merge_keys, how="left")
    path_ghg_processed.mkdir(parents=True, exist_ok=True)
    inventories_subnat[gas].to_csv(path_ghg_processed / f"inventory_subnat_{gas}.csv", index=None)

    # Shares of total world sector emissions
    sectors_wld_total = inventories[gas][["jurisdiction", "year", "ipcc_code", "iea_code", gas]].groupby(["ipcc_code", "year"]).sum()
    sectors_wld_total.reset_index(inplace=True)

    sectors_wld_total.drop(["jurisdiction", "iea_code"], axis=1, inplace=True)

    inventories_subnat_wldSect[gas] = ecp_inv_share.emissions_share_wld_sectors(inventories_subnat[gas], sectors_wld_total, gas, "subnational")

    # Shares of total country-sector emissions
    sectors_ctry_total = inventories[gas][["jurisdiction", "year", "ipcc_code", "iea_code", gas]].groupby(["jurisdiction", "year", "ipcc_code"]).sum()
    sectors_ctry_total.reset_index(inplace=True)

    inventories_subnat_ctrySect[gas] = ecp_inv_share.emissions_share_ctry_sectors(inventories_subnat[gas], sectors_ctry_total, gas)


CO2


CH4


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_inventory_share_func.py:180: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[share_col] = merged[gas].astype(float).div(denom.replace({0: np.nan}))


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_inventory_share_func.py:250: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[share_col] = merged[gas].astype(float).div(merged[denom_col].replace({0: np.nan}))


N2O


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_inventory_share_func.py:180: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[share_col] = merged[gas].astype(float).div(denom.replace({0: np.nan}))


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_inventory_share_func.py:250: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged[share_col] = merged[gas].astype(float).div(merged[denom_col].replace({0: np.nan}))


In [14]:
for gas in gases:
    subnat_processed_dir = path_ghg_processed / "subnational" / gas
    subnat_sector_dir = subnat_processed_dir / "sector_level"
    subnat_processed_dir.mkdir(parents=True, exist_ok=True)
    subnat_sector_dir.mkdir(parents=True, exist_ok=True)

    inventories_subnat_ctrySect[gas].to_csv(subnat_sector_dir / f"inventory_{gas}.csv", index=None)

    for jur in subnat_dic.keys():
        inventories_subnat[gas].loc[inventories_subnat[gas].jurisdiction==jur, :].to_csv(
            subnat_processed_dir / f"inventory_{gas}_{subnat_dic[jur]}.csv",
            index=None,
        )


In [15]:
lastInvYear["subnat"] = infer_subnat_last_inv_year(inventories_subnat, subnat_lists)
lastInvYear["subnat"]


{'United States': 2021,
 'Canada': 2023,
 'China': 2018,
 'Mexico': 2024,
 'Japan': 2024}

# Coverage 
## I. Disaggregated coverage dataframes

** Note: National and subnational inventories do not have the same level of disaggregation **

In [16]:
coverage_nat = {}
coverage_subnat = {}
coverage = {}
coverage_sect = {}
coverage_nat_sect = {}
coverage_subnat_sect = {}

# Canada inventory paths need updating to bring subnat start year to 2021

# SHARE OF JURISDICTIONS TOTAL EMISSIONS
for gas in gases:
    print(gas)
    coverage_nat[gas] = ecp_coverage.coverage(inventories[gas], lastInvYear["national"], lastDbYear, wcpd[gas], gas,
                                              False, "national")
    coverage_subnat[gas] = ecp_coverage.coverage(inventories_subnat[gas], lastInvYear["subnat"], lastDbYear, wcpd[gas], gas,
                                                 False, "subnational")

    coverage_all = pd.concat([coverage_nat[gas], coverage_subnat[gas]], ignore_index=True)
    coverage_all = coverage_all.loc[coverage_all["jurisdiction"]!="World", :]

    # Coverage figures should be calculated only based on aggregation of the most disaggregated flows, not their higher-level aggregation. 
    # Otherwise this might result in double counting. Hence aggregate sectors should be dropped from coverage dataframe.
    # It also currently excludes coverage of international aviation ('ABFLOW039') and marine ('ABFLOW040') bunkers 
    # as they are currently excluded from national total emissions.
    # Drop combustion sectors that are aggregation of lower level sectors and concatenate all coverage dataframes into a single one*

    flow_excl = ['1A', '1A1A', '1A1C', '1A2', '1A3'] #'1A1C' is exluded here as ABFLOW011 emissions are attributed twice (to both 1A1B and 1A1C)
    coverage_all = coverage_all.loc[~coverage_all.ipcc_code.isin(flow_excl), :]

    coverage[gas] = coverage_all


    # SHARE OF SECTORS' GLOBAL TOTAL EMISSIONS

    coverage_nat_sect[gas] = ecp_coverage.coverage(inventories_wldSect[gas], lastInvYear["national"], lastDbYear, wcpd[gas], gas,
                                True, "national")
    coverage_subnat_sect[gas] = ecp_coverage.coverage(inventories_subnat_wldSect[gas], lastInvYear["subnat"], lastDbYear, wcpd[gas], gas,
                                    True, "subnational")

    coverage_sect_all = pd.concat([coverage_nat_sect[gas], coverage_subnat_sect[gas]])
    coverage_sect_all = coverage_sect_all.loc[coverage_sect_all["jurisdiction"]!="World", :]

    coverage_sect[gas] = coverage_sect_all

CO2


['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']
['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']


['iea_code', 'ipcc_code', 'jurisdiction', 'year']
['iea_code', 'ipcc_code', 'jurisdiction', 'year']


['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']
['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']


['iea_code', 'ipcc_code', 'jurisdiction', 'year']
['iea_code', 'ipcc_code', 'jurisdiction', 'year']


...

N2O


['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']
['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']


['iea_code', 'ipcc_code', 'jurisdiction', 'year']
['iea_code', 'ipcc_code', 'jurisdiction', 'year']


['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']
['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year']


['iea_code', 'ipcc_code', 'jurisdiction', 'year']
['iea_code', 'ipcc_code', 'jurisdiction', 'year']


In [17]:
gas = "CH4"

print(inventories[gas].loc[(inventories[gas].jurisdiction=="Singapore") & (inventories[gas].year==2022) & (~inventories[gas].CH4.isna()), 
                    ['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year', gas]])


#coverage[gas].loc[(coverage[gas].jurisdiction=="Poland") & (coverage[gas].ipcc_code=="1A1A1"), 
#                    ['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year', 'ets_id', 'tax_id', 'cov_tax_CH4_jurCH4']]

print(wcpd[gas].loc[(wcpd[gas].jurisdiction=="Singapore") & (wcpd[gas].year==2022) & (~wcpd[gas].tax_id.isna()),
              ['Product', 'iea_code', 'ipcc_code', 'jurisdiction', 'year', 'ets_id', 'tax_id', 'tax_rate_excl_ex_clcu', 'tax_ex_rate', 'tax_rate_incl_ex_clcu']])

             Product   iea_code ipcc_code jurisdiction  year          CH4
1599675  Natural gas  ABFLOW001        1A    Singapore  2022         12.4
1599676  Natural gas  ABFLOW001        1A    Singapore  2022         12.4
1599679          Oil  ABFLOW001        1A    Singapore  2022         44.7
1599680          Oil  ABFLOW001        1A    Singapore  2022         44.7
1599686         Coal  ABFLOW002      1A1A    Singapore  2022          0.3
1599687  Natural gas  ABFLOW002      1A1A    Singapore  2022          9.2
1599688          Oil  ABFLOW002      1A1A    Singapore  2022          1.1
1599700          Oil  ABFLOW011      1A1B    Singapore  2022          3.1
1599704         Coal  ABFLOW012       1A2    Singapore  2022          2.3
1599705  Natural gas  ABFLOW012       1A2    Singapore  2022          1.6
1599706          Oil  ABFLOW012       1A2    Singapore  2022          5.7
1599747  Natural gas  ABFLOW027       1A3    Singapore  2022          0.4
1599748          Oil  ABFLOW027       

             Product   iea_code ipcc_code jurisdiction  year ets_id   tax_id  \
3206793         Coal  ABFLOW003     1A1A1    Singapore  2022    NaN  sgp_tax   
3206794  Natural gas  ABFLOW003     1A1A1    Singapore  2022    NaN  sgp_tax   
3206795          Oil  ABFLOW003     1A1A1    Singapore  2022    NaN  sgp_tax   
3206796         Coal  ABFLOW004     1A1A2    Singapore  2022    NaN  sgp_tax   
3206797  Natural gas  ABFLOW004     1A1A2    Singapore  2022    NaN  sgp_tax   
...              ...        ...       ...          ...   ...    ...      ...   
3206996           NA         NA       2D1    Singapore  2022    NaN  sgp_tax   
3206997           NA         NA       2D2    Singapore  2022    NaN  sgp_tax   
3206998           NA         NA       2D3    Singapore  2022    NaN  sgp_tax   
3206999           NA         NA       2D4    Singapore  2022    NaN  sgp_tax   
3207000           NA         NA        2E    Singapore  2022    NaN  sgp_tax   

         tax_rate_excl_ex_clcu  tax_ex_

## II. Aggregate coverage

- "The sum over all pricing mechanisms" of [emissions_share x coverage_factor] minus the overlapping coverage

We account for the fact that more than one tax scheme or ets scheme can apply to the same emissions. However, covered emissions should be counted only once when covered by one or more scheme. To calculate overlapping coverage at the sector-fuel level, we use the `overlap_` variable in `wcpd_all` dataframe created above.

### II.1 jurisdictions

In [18]:
agg_cov = {}

for gas in gases:
    coverage_all_gas = coverage[gas]

    # Initialize output DataFrame
    coverage_agg = coverage_all_gas[["jurisdiction", "year", "ipcc_code", "iea_code", "Product"]].copy()

    # Define coverage scopes
    scopes = ["jurGHG", f"jur{gas}", "wldGHG", f"wld{gas}", "supraGHG", f"supra{gas}"]

    # Utility function to get relevant columns
    def get_cov_columns(prefix, scopes, exclude_overlap=True, rename_prefix=None):
        result = {}
        for scope in scopes:
            matching = [
                col for col in coverage_all_gas.columns
                if prefix in col and scope in col and (not exclude_overlap or "overlap" not in col)
            ]
            if rename_prefix:
                key = f"{rename_prefix}_{gas}_{scope}"
            else:
                key = f"{prefix}_{gas}_{scope}"
            result[key] = matching
        return result

    # Extract columns by instrument
    tax_columns = get_cov_columns("cov_tax", scopes)
    ets_columns = get_cov_columns("cov_ets", scopes)
    all_columns = get_cov_columns("cov_", scopes, exclude_overlap=True, rename_prefix="cov_all")

    # Overlap columns
    all_overlap_dic = {
        f"cov_all_{gas}_{scope}": f"cov_overlap_{gas}_{scope}" for scope in scopes
    }

    # A. Sum across tax and ETS instruments
    for dic in [tax_columns, ets_columns]:
        for new_col, cols in dic.items():
            coverage_agg[new_col] = coverage_all_gas[cols].sum(axis=1)

    # B. Sum across all instruments and subtract overlaps
    for new_col, cols in all_columns.items():
        coverage_agg[new_col] = coverage_all_gas[cols].sum(axis=1)
        overlap_col = all_overlap_dic.get(new_col)
        if overlap_col in coverage_all_gas.columns:
            coverage_agg[new_col] -= coverage_all_gas[overlap_col]

    # C. Aggregate across emission categories (rows)
    agg = coverage_agg.groupby(["jurisdiction", "year"], as_index=False).sum(numeric_only=True)

    # Store in dictionary
    agg_cov[gas] = agg


In [19]:
# WORLD TOTAL COVERAGE 

for gas in gases:
    cov_world_agg = agg_cov[gas][["jurisdiction","year", "cov_tax_"+gas+"_wld"+gas, "cov_ets_"+gas+"_wld"+gas, 
                                        "cov_tax_"+gas+"_wldGHG", "cov_ets_"+gas+"_wldGHG"]]

    cov_world_agg.reset_index(inplace=True)
    cov_world_agg = cov_world_agg.groupby(['year']).sum()

    cov_world_agg["cov_all_"+gas+"_jurGHG"] = cov_world_agg["cov_tax_"+gas+"_wldGHG"] + cov_world_agg["cov_ets_"+gas+"_wldGHG"]
    cov_world_agg["cov_all_"+gas+"_jur"+gas] = cov_world_agg["cov_tax_"+gas+"_wld"+gas] + cov_world_agg["cov_ets_"+gas+"_wld"+gas]
    
    cov_world_agg["cov_all_"+gas+"_wldGHG"] = cov_world_agg["cov_all_"+gas+"_jurGHG"]
    cov_world_agg["cov_all_"+gas+"_wld"+gas] = cov_world_agg["cov_all_"+gas+"_jur"+gas]

    # addind values in 'jur' columns for the "World" jurisdiction
    cov_world_agg["cov_tax_"+gas+"_jurGHG"] = cov_world_agg["cov_tax_"+gas+"_wldGHG"]
    cov_world_agg["cov_ets_"+gas+"_jurGHG"] = cov_world_agg["cov_ets_"+gas+"_wldGHG"]
    cov_world_agg["cov_tax_"+gas+"_jur"+gas] = cov_world_agg["cov_tax_"+gas+"_wld"+gas]
    cov_world_agg["cov_ets_"+gas+"_jur"+gas] = cov_world_agg["cov_ets_"+gas+"_wld"+gas]

#    cov_world_agg["cov_tax_"+gas+"_wldGHG"] = "NA"
#    cov_world_agg["cov_ets_"+gas+"_wldGHG"] = "NA"
#    cov_world_agg["cov_tax_"+gas+"_wld"+gas] = "NA"
#    cov_world_agg["cov_ets_"+gas+"_wld"+gas] = "NA"

    cov_world_agg["jurisdiction"] = "World"

    cov_world_agg.drop("index", axis=1, inplace=True)
    cov_world_agg.reset_index(inplace=True)

    agg_cov[gas] = pd.concat([agg_cov[gas], cov_world_agg])


In [20]:
# National-level coverage from subnational schemes

for gas in gases:

      for subnat_list in subnat_lists.keys():
            temp = agg_cov[gas].loc[agg_cov[gas].jurisdiction.isin(subnat_lists[subnat_list]), :]
            temp = temp.groupby(["year"]).sum()
            temp.reset_index(inplace=True)
            temp["jurisdiction"] = subnat_list+"_sub" # indicating it is the country-level coverage from subnational mechanisms

            temp[["cov_tax_"+gas+"_jurGHG", "cov_tax_"+gas+"_jur"+gas, "cov_ets_"+gas+"_jurGHG", "cov_ets_"+gas+"_jur"+gas,
                  "cov_all_"+gas+"_jurGHG", "cov_all_"+gas+"_jur"+gas]] = np.nan
            
            swap_list = {"cov_tax_"+gas+"_jurGHG":"cov_tax_"+gas+"_supraGHG", "cov_tax_"+gas+"_jur"+gas:"cov_tax_"+gas+"_supra"+gas, "cov_ets_"+gas+"_jurGHG":"cov_ets_"+gas+"_supraGHG", 
                        "cov_ets_"+gas+"_jur"+gas:"cov_ets_"+gas+"_supra"+gas, "cov_all_"+gas+"_jurGHG":"cov_all_"+gas+"_supraGHG", "cov_all_"+gas+"_jur"+gas:"cov_all_"+gas+"_supra"+gas,
                        "cov_tax_"+gas+"_supraGHG":"cov_tax_"+gas+"_jurGHG", "cov_tax_"+gas+"_supra"+gas:"cov_tax_"+gas+"_jur"+gas, "cov_ets_"+gas+"_supraGHG":"cov_ets_"+gas+"_jurGHG", 
                        "cov_ets_"+gas+"_supra"+gas:"cov_ets_"+gas+"_jur"+gas, "cov_all_"+gas+"_supraGHG":"cov_all_"+gas+"_jurGHG", "cov_all_"+gas+"_supra"+gas:"cov_all_"+gas+"_jur"+gas}
            
            temp.rename(columns=swap_list, inplace=True)
            
            temp_nat = agg_cov[gas].loc[agg_cov[gas].jurisdiction == subnat_list, :]

            temp_nat_subnat = pd.concat([temp_nat, temp])
            temp_nat_subnat = temp_nat_subnat.groupby(["year"]).sum() # summing country-level coverage from country-level and subnational mechanisms
            temp_nat_subnat.reset_index(inplace=True)

            temp_nat_subnat["jurisdiction"] = subnat_list

            agg_cov[gas] = agg_cov[gas].loc[agg_cov[gas].jurisdiction != subnat_list, :]
            
            agg_cov[gas] = pd.concat([agg_cov[gas], temp_nat_subnat])
    

In [21]:
# NA values for all entries of 'supra' columns of national jurisdictions

for gas in gases:

    supra_cols = ["cov_tax_"+gas+"_supraGHG", "cov_tax_"+gas+"_supra"+gas, "cov_ets_"+gas+"_supraGHG", 
                "cov_ets_"+gas+"_supra"+gas, "cov_all_"+gas+"_supraGHG", "cov_all_"+gas+"_supra"+gas]

    agg_cov[gas].loc[~agg_cov[gas].jurisdiction.isin(all_subnat_list), supra_cols] = np.nan

    coverage_outdir = path_dataset_output / "coverage" / "jurisdictions"
    coverage_outdir.mkdir(parents=True, exist_ok=True)
    coverage_agg_OUT = agg_cov[gas].fillna("NA")
    coverage_agg_OUT.sort_values(by=["jurisdiction", "year"]).to_csv(coverage_outdir / f"tot_coverage_jurisdiction_{gas}_{d1}.csv", index=None)

/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2419664913.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  coverage_agg_OUT = agg_cov[gas].fillna("NA")


### II.2 World sectors

In [22]:
coverage_WldSect = {}

for gas in gases:
    
    coverage_sect[gas]

    cov_tax_columns_WldSectGas = [x for x in coverage_sect[gas].columns if "cov_tax" in x and "wld_sect" in x]
    cov_ets_columns_WldSectGas = [x for x in coverage_sect[gas].columns if "cov_ets" in x and "wld_sect" in x]
    cov_all_columns_WldSectGas = [x for x in coverage_sect[gas].columns if "cov_" in x and "wld_sect" in x]

    tax_columns = {"cov_tax_"+gas+"_WldSect"+gas:cov_tax_columns_WldSectGas}
    ets_columns = {"cov_ets_"+gas+"_WldSect"+gas:cov_ets_columns_WldSectGas}
    all_columns = {"cov_all_"+gas+"_WldSect"+gas:cov_all_columns_WldSectGas}

    coverage_sect_agg_schemes = coverage_sect[gas][["jurisdiction", "year", "ipcc_code", "iea_code", "Product"]]

    for dic in [tax_columns, ets_columns, all_columns]:
        for key in dic.keys():
            coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)

    coverage_WldSect[gas] = coverage_sect_agg_schemes.groupby(['ipcc_code','year']).sum()
    coverage_WldSect[gas].reset_index(inplace=True)

    coverage_wld_outdir = path_dataset_output / "coverage" / "world_sectors"
    coverage_wld_outdir.mkdir(parents=True, exist_ok=True)
    coverage_WldSect[gas].to_csv(coverage_wld_outdir / f"tot_coverage_world_sectors_{gas}_{d1}.csv", index=None)

/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)
/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)
/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)
/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/2345438821.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  coverage_sect_agg_schemes[key] = coverage_sect[gas][dic[key]].sum(axis=1)


# Emissions-weighted Carbon Price (ECP)
Combines: (i) (total) coverage of ETS and associated price, (ii) user-fuel coverage of taxes and associated tax rates


In [23]:
prices_usd = {}

for gas in gases:
    # simply execute function to create cFlxRate series
    ecp_cur_conv.cur_conv(wcpd[gas], gas, subnat_lists["Canada"], subnat_lists["United States"], subnat_lists["China"], False, None)

    wcpd_usd = ecp_cur_conv.cur_conv(wcpd[gas], gas, subnat_lists["Canada"], subnat_lists["United States"], subnat_lists["China"], True, 2021)

    #Bring together calculated emissions share at sector and sector-fuel level and carbon prices in constant USD

    id_columns = [x for x in wcpd_usd.columns if bool(re.match(re.compile("ets.+id"), x))==True or bool(re.match(re.compile("tax.+id"), x))==True]
    price_columns = [x for x in wcpd_usd.columns if bool(re.match(re.compile("ets.+price_usd_k"), x))==True or bool(re.match(re.compile("tax.+rate.+usd_k"), x))==True]

    prices_usd[gas] = wcpd_usd[['jurisdiction', 'year', 'ipcc_code', 'iea_code', 'Product']+price_columns]

    prices_usd[gas]['Product'].fillna('NA', inplace=True)
    prices_usd[gas]['iea_code'].fillna('NA', inplace=True)

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_gen_func.py:198: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["jurisdiction"].replace({"Czechia": "Czech Republic"}, inplace=True)


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_gen_func.py:198: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["jurisdiction"].replace({"Czechia": "Czech Republic"}, inplace=True)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/3077147567.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  prices_usd[gas]['Product'].fillna('NA', inplace=True)
/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/3077147567.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prices_usd[gas]['Product'].fillna

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_gen_func.py:198: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["jurisdiction"].replace({"Czechia": "Czech Republic"}, inplace=True)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/3077147567.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  prices_usd[gas]['Product'].fillna('NA', inplace=True)
/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/3077147567.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prices_usd[gas]['Product'].fillna

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_gen_func.py:198: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["jurisdiction"].replace({"Czechia": "Czech Republic"}, inplace=True)


/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/3077147567.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  prices_usd[gas]['Product'].fillna('NA', inplace=True)
/var/folders/c4/8ngnh6t562559dgckm_h844r0000gn/T/ipykernel_83364/3077147567.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  prices_usd[gas]['Product'].fillna

## I. ECP from ETS and taxes (time-varying and fixed weights, jurisdiction level)

National and subnational jurisdictions, sectoral level

In [24]:
ecp_variables_map = {}

ecp_tv = {}
ecp_fixed = {}

for gas in gases:
    ecp_tv_nat = ecp_wav.ecp(coverage_nat[gas], prices_usd[gas], "national", gas, flow_excl, "time_varying", sectors=False)
    ecp_tv_subnat = ecp_wav.ecp(coverage_subnat[gas], prices_usd[gas], "subnational", gas, flow_excl, "time_varying", sectors=False)
    
    ecp_tv[gas] = pd.concat([ecp_tv_nat, ecp_tv_subnat])

    ecp_fixed_nat = ecp_wav.ecp(coverage_nat[gas], prices_usd[gas], "national", gas, flow_excl, "fixed", 2015, sectors=False)
    ecp_fixed_subnat = ecp_wav.ecp(coverage_subnat[gas], prices_usd[gas], "subnational", gas, flow_excl, "fixed", 2015, sectors=False)
    
    ecp_fixed[gas] = pd.concat([ecp_fixed_nat, ecp_fixed_subnat])


Sector ECP mapping:
ecp_ets_jurGHG_usd_k: 4 columns → ['cov_ets_2_CO2_jurGHG', 'cov_ets_CO2_jurGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_jurCO2_usd_k: 4 columns → ['cov_ets_2_CO2_jurCO2', 'cov_ets_CO2_jurCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldGHG_usd_k: 4 columns → ['cov_ets_2_CO2_wldGHG', 'cov_ets_CO2_wldGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldCO2_usd_k: 4 columns → ['cov_ets_2_CO2_wldCO2', 'cov_ets_CO2_wldCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_jurGHG_usd_k: 2 columns → ['cov_tax_CO2_jurGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_jurCO2_usd_k: 2 columns → ['cov_tax_CO2_jurCO2', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldGHG_usd_k: 2 columns → ['cov_tax_CO2_wldGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldCO2_usd_k: 2 columns → ['cov_tax_CO2_wldCO2', 'tax_rate_incl_ex_usd_k']


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_weightedAverage.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df[output_cols].fillna(0)



Sector ECP mapping:
ecp_ets_jurGHG_usd_k: 4 columns → ['cov_ets_2_CO2_jurGHG', 'cov_ets_CO2_jurGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_jurCO2_usd_k: 4 columns → ['cov_ets_2_CO2_jurCO2', 'cov_ets_CO2_jurCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldGHG_usd_k: 4 columns → ['cov_ets_2_CO2_wldGHG', 'cov_ets_CO2_wldGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldCO2_usd_k: 4 columns → ['cov_ets_2_CO2_wldCO2', 'cov_ets_CO2_wldCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_jurGHG_usd_k: 2 columns → ['cov_tax_CO2_jurGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_jurCO2_usd_k: 2 columns → ['cov_tax_CO2_jurCO2', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldGHG_usd_k: 2 columns → ['cov_tax_CO2_wldGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldCO2_usd_k: 2 columns → ['cov_tax_CO2_wldCO2', 'tax_rate_incl_ex_usd_k']
ecp_ets_supraGHG_usd_k: 4 columns → ['cov_ets_2_CO2_supraGHG', 'cov_ets_CO2_supraGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_supraCO2_usd_k: 4 columns → 

/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_weightedAverage.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df[output_cols].fillna(0)



Sector ECP mapping:
ecp_ets_jurGHG_usd_k: 4 columns → ['cov_ets_2_CO2_jurGHG', 'cov_ets_CO2_jurGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_jurCO2_usd_k: 4 columns → ['cov_ets_2_CO2_jurCO2', 'cov_ets_CO2_jurCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldGHG_usd_k: 4 columns → ['cov_ets_2_CO2_wldGHG', 'cov_ets_CO2_wldGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldCO2_usd_k: 4 columns → ['cov_ets_2_CO2_wldCO2', 'cov_ets_CO2_wldCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']


...


Sector ECP mapping:
ecp_ets_jurGHG_usd_k: 4 columns → ['cov_ets_2_N2O_jurGHG', 'cov_ets_N2O_jurGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_jurN2O_usd_k: 4 columns → ['cov_ets_2_N2O_jurN2O', 'cov_ets_N2O_jurN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldGHG_usd_k: 4 columns → ['cov_ets_2_N2O_wldGHG', 'cov_ets_N2O_wldGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wldN2O', 'cov_ets_N2O_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_jurGHG_usd_k: 2 columns → ['cov_tax_N2O_jurGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_jurN2O_usd_k: 2 columns → ['cov_tax_N2O_jurN2O', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldGHG_usd_k: 2 columns → ['cov_tax_N2O_wldGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldN2O_usd_k: 2 columns → ['cov_tax_N2O_wldN2O', 'tax_rate_incl_ex_usd_k']



Sector ECP mapping:
ecp_ets_jurGHG_usd_k: 4 columns → ['cov_ets_2_N2O_jurGHG', 'cov_ets_N2O_jurGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_jurN2O_usd_k: 4 columns → ['cov_ets_2_N2O_jurN2O', 'cov_ets_N2O_jurN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldGHG_usd_k: 4 columns → ['cov_ets_2_N2O_wldGHG', 'cov_ets_N2O_wldGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wldN2O', 'cov_ets_N2O_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_jurGHG_usd_k: 2 columns → ['cov_tax_N2O_jurGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_jurN2O_usd_k: 2 columns → ['cov_tax_N2O_jurN2O', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldGHG_usd_k: 2 columns → ['cov_tax_N2O_wldGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldN2O_usd_k: 2 columns → ['cov_tax_N2O_wldN2O', 'tax_rate_incl_ex_usd_k']
ecp_ets_supraGHG_usd_k: 4 columns → ['cov_ets_2_N2O_supraGHG', 'cov_ets_N2O_supraGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_supraN2O_usd_k: 4 columns → 


Sector ECP mapping:
ecp_ets_jurGHG_usd_k: 4 columns → ['cov_ets_2_N2O_jurGHG', 'cov_ets_N2O_jurGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_jurN2O_usd_k: 4 columns → ['cov_ets_2_N2O_jurN2O', 'cov_ets_N2O_jurN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldGHG_usd_k: 4 columns → ['cov_ets_2_N2O_wldGHG', 'cov_ets_N2O_wldGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wldN2O', 'cov_ets_N2O_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']


ecp_tax_jurGHG_usd_k: 2 columns → ['cov_tax_N2O_jurGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_jurN2O_usd_k: 2 columns → ['cov_tax_N2O_jurN2O', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldGHG_usd_k: 2 columns → ['cov_tax_N2O_wldGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldN2O_usd_k: 2 columns → ['cov_tax_N2O_wldN2O', 'tax_rate_incl_ex_usd_k']



Sector ECP mapping:
ecp_ets_jurGHG_usd_k: 4 columns → ['cov_ets_2_N2O_jurGHG', 'cov_ets_N2O_jurGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_jurN2O_usd_k: 4 columns → ['cov_ets_2_N2O_jurN2O', 'cov_ets_N2O_jurN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldGHG_usd_k: 4 columns → ['cov_ets_2_N2O_wldGHG', 'cov_ets_N2O_wldGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_wldN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wldN2O', 'cov_ets_N2O_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_jurGHG_usd_k: 2 columns → ['cov_tax_N2O_jurGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_jurN2O_usd_k: 2 columns → ['cov_tax_N2O_jurN2O', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldGHG_usd_k: 2 columns → ['cov_tax_N2O_wldGHG', 'tax_rate_incl_ex_usd_k']
ecp_tax_wldN2O_usd_k: 2 columns → ['cov_tax_N2O_wldN2O', 'tax_rate_incl_ex_usd_k']
ecp_ets_supraGHG_usd_k: 4 columns → ['cov_ets_2_N2O_supraGHG', 'cov_ets_N2O_supraGHG', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_ets_supraN2O_usd_k: 4 columns → 

In [25]:
ecp_tv_sect = {}
ecp_fixed_sect = {}

for gas in gases:
    ecp_tv_nat_sect = ecp_wav.ecp(coverage_nat_sect[gas], prices_usd[gas], "national", gas, flow_excl, "time_varying", sectors=True)
    ecp_tv_subnat_sect = ecp_wav.ecp(coverage_subnat_sect[gas], prices_usd[gas], "subnational", gas, flow_excl, "time_varying", sectors=True)
    
    ecp_tv_sect[gas] = pd.concat([ecp_tv_nat_sect, ecp_tv_subnat_sect])
    ecp_world_sect_outdir = path_dataset_output / "ecp" / "ecp_world_sectors"
    ecp_world_sect_outdir.mkdir(parents=True, exist_ok=True)
    ecp_tv_nat_sect.groupby(["ipcc_code", "year"]).sum().to_csv(ecp_world_sect_outdir / f"world_sectoral_ecp_{gas}_{d1}.csv")

    ecp_fixed_nat_sect = ecp_wav.ecp(coverage_nat_sect[gas], prices_usd[gas], "national", gas, flow_excl, "fixed", 2015, sectors=True)
    ecp_fixed_subnat_sect = ecp_wav.ecp(coverage_subnat_sect[gas], prices_usd[gas], "subnational", gas, flow_excl, "fixed", 2015, sectors=True)
    
    ecp_fixed_sect[gas] = pd.concat([ecp_fixed_nat_sect, ecp_fixed_subnat_sect])


Sector ECP mapping:
ecp_ets_sectCO2_usd_k: 4 columns → ['cov_ets_2_CO2_wld_sect_wldCO2', 'cov_ets_CO2_wld_sect_wldCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectCO2_usd_k: 2 columns → ['cov_tax_CO2_wld_sect_wldCO2', 'tax_rate_incl_ex_usd_k']


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_weightedAverage.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df[output_cols].fillna(0)



Sector ECP mapping:
ecp_ets_sectCO2_usd_k: 4 columns → ['cov_ets_2_CO2_wld_sect_wldCO2', 'cov_ets_CO2_wld_sect_wldCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectCO2_usd_k: 2 columns → ['cov_tax_CO2_wld_sect_wldCO2', 'tax_rate_incl_ex_usd_k']


/Users/geoffroydolphin/GitHub/ECP/_code/compilation/_utils/dep_ecp/ecp_v3_weightedAverage.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = temp_df[output_cols].fillna(0)



Sector ECP mapping:
ecp_ets_sectCO2_usd_k: 4 columns → ['cov_ets_2_CO2_wld_sect_wldCO2', 'cov_ets_CO2_wld_sect_wldCO2', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectCO2_usd_k: 2 columns → ['cov_tax_CO2_wld_sect_wldCO2', 'tax_rate_incl_ex_usd_k']


...


Sector ECP mapping:
ecp_ets_sectCH4_usd_k: 4 columns → ['cov_ets_2_CH4_wld_sect_wldCH4', 'cov_ets_CH4_wld_sect_wldCH4', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectCH4_usd_k: 2 columns → ['cov_tax_CH4_wld_sect_wldCH4', 'tax_rate_incl_ex_usd_k']



Sector ECP mapping:
ecp_ets_sectN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wld_sect_wldN2O', 'cov_ets_N2O_wld_sect_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectN2O_usd_k: 2 columns → ['cov_tax_N2O_wld_sect_wldN2O', 'tax_rate_incl_ex_usd_k']



Sector ECP mapping:
ecp_ets_sectN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wld_sect_wldN2O', 'cov_ets_N2O_wld_sect_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectN2O_usd_k: 2 columns → ['cov_tax_N2O_wld_sect_wldN2O', 'tax_rate_incl_ex_usd_k']



Sector ECP mapping:
ecp_ets_sectN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wld_sect_wldN2O', 'cov_ets_N2O_wld_sect_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectN2O_usd_k: 2 columns → ['cov_tax_N2O_wld_sect_wldN2O', 'tax_rate_incl_ex_usd_k']



Sector ECP mapping:
ecp_ets_sectN2O_usd_k: 4 columns → ['cov_ets_2_N2O_wld_sect_wldN2O', 'cov_ets_N2O_wld_sect_wldN2O', 'ets_price_usd_k', 'ets_2_price_usd_k']
ecp_tax_sectN2O_usd_k: 2 columns → ['cov_tax_N2O_wld_sect_wldN2O', 'tax_rate_incl_ex_usd_k']


In [26]:
ecp_tv_agg = {}
ecp_fixed_agg = {}

for gas in gases: 
    ecp_tv_agg[gas] = ecp_wav.ecp_aggregation(ecp_tv[gas], gas)
    ecp_fixed_agg[gas] = ecp_wav.ecp_aggregation(ecp_fixed[gas], gas)

    # National-level ecp from subnational schemes        

    for key in subnat_lists.keys():
        ecp_tv_agg[gas] = ecp_wav.national_from_subnat(ecp_tv_agg[gas], subnat_lists[key], key, gas)
        ecp_fixed_agg[gas] = ecp_wav.national_from_subnat(ecp_fixed_agg[gas], subnat_lists[key], key, gas)

    # NA values for all entries of 'supra' columns of national jurisdictions

    supra_cols = ["ecp_ets_supraGHG_usd_k", "ecp_tax_supraGHG_usd_k", 
                  "ecp_ets_supra"+gas+"_usd_k", "ecp_tax_supra"+gas+"_usd_k", 
                  "ecp_all_supraGHG_usd_k", "ecp_all_supra"+gas+"_usd_k"]

    for df in [ecp_tv_agg[gas], ecp_fixed_agg[gas]]:
        df.loc[~df.jurisdiction.isin(all_subnat_list), supra_cols] = np.nan

In [27]:
ecp_tv_outdir = path_dataset_output / "ecp" / "ecp_economy" / "ecp_vw"
ecp_fixed_outdir = path_dataset_output / "ecp" / "ecp_economy" / "ecp_fw"
ecp_tv_outdir.mkdir(parents=True, exist_ok=True)
ecp_fixed_outdir.mkdir(parents=True, exist_ok=True)

for gas in gases:
    col_sel = ["jurisdiction", "year", 
                "ecp_ets_jurGHG_usd_k", "ecp_tax_jurGHG_usd_k", "ecp_all_jurGHG_usd_k", 
                "ecp_ets_jur"+gas+"_usd_k", "ecp_tax_jur"+gas+"_usd_k", "ecp_all_jur"+gas+"_usd_k",
                "ecp_ets_supraGHG_usd_k", "ecp_tax_supraGHG_usd_k", "ecp_all_supraGHG_usd_k", 
                "ecp_ets_supra"+gas+"_usd_k", "ecp_tax_supra"+gas+"_usd_k", "ecp_all_supra"+gas+"_usd_k"]

    ecp_tv_agg[gas][col_sel].fillna("NA").sort_values(by=["jurisdiction", "year"]).to_csv(ecp_tv_outdir / f"ecp_tv_{gas}_{d1}.csv", index=None)
    ecp_fixed_agg[gas][col_sel].fillna("NA").sort_values(by=["jurisdiction", "year"]).to_csv(ecp_fixed_outdir / f"ecp_fixed_{gas}_{d1}.csv", index=None)

## II. Calculation of ECP from ETS and taxes (CO2 only, constant, jurisdiction-specific weights, jurisdiction level)